In [1]:
import requests
import pandas as pd
import json
import time

Market Data Exploration

In [2]:
GAMMA_API = "https://gamma-api.polymarket.com"

In [3]:
response = requests.get(
    f"{GAMMA_API}/markets",
    params={
        "limit": 1000,
        "offset": 0
    },
    timeout=30
)

print("Status:", response.status_code)
print("Response time:", response.elapsed.total_seconds(), "seconds")

markets = response.json()

print("Markets returned:", len(markets))

Status: 200
Response time: 0.475694 seconds
Markets returned: 100


In [4]:
all_markets = []

for offset in range(0, 1000, 100):
    response = requests.get(
        f"{GAMMA_API}/markets",
        params={
            "limit": 100,
            "offset": offset
        },
        timeout=30
    )

    response.raise_for_status()

    batch = response.json()
    all_markets.extend(batch)

    print(f"Offset {offset}: {len(batch)} markets")

    if len(batch) < 100:
        break

    time.sleep(0.2)

print("Total markets collected:", len(all_markets))

Offset 0: 100 markets
Offset 100: 100 markets
Offset 200: 100 markets
Offset 300: 100 markets
Offset 400: 100 markets
Offset 500: 100 markets
Offset 600: 100 markets
Offset 700: 100 markets
Offset 800: 100 markets
Offset 900: 100 markets
Total markets collected: 1000


In [5]:
markets_df = pd.DataFrame(all_markets)

print("Shape:", markets_df.shape)

markets_df.head()

Shape: (1000, 101)


,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,negRiskMarketID,seriesColor,showGmpSeries,showGmpOutcome,oneHourPriceChange,umaResolutionStatus,eventStartTime,gameStartTime,secondsDelay,groupItemRange
0,559651,Xi Jinping out before 2027?,0xa467b14d51f01b957109d9cbb1d6c124fab2a089d52e...,xi-jinping-out-before-2027,,2027-01-01T04:59:00Z,283154.74182,2025-07-03T20:37:00.228Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,559652,Will Gavin Newsom win the 2028 Democratic pres...,0x0f49db97f71c68b1e42a6d16e3de93d85dbf7d4148e3...,will-gavin-newsom-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,299000.44598,2025-07-11T18:35:56.805Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN,NaN
2,559653,Will Alexandria Ocasio-Cortez win the 2028 Dem...,0xe6bcc2f1dd025ce5e1833190f7c60a71171c94f805df...,will-alexandria-ocasio-cortez-win-the-2028-dem...,,2028-11-07T00:00:00Z,334269.35073,2025-07-11T18:35:59.075Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN,NaN
3,559654,Will Pete Buttigieg win the 2028 Democratic pr...,0x4c325469d9b516ef4e6b8f73a81a12607dec075e3c2f...,will-pete-buttigieg-win-the-2028-democratic-pr...,,2028-11-07T00:00:00Z,369176.63014,2025-07-11T18:35:58.818Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN,NaN
4,559655,Will Josh Shapiro win the 2028 Democratic pres...,0xd65891729ce093cc12236856837eba1a0872fc7998fd...,will-josh-shapiro-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,261367.22332,2025-07-11T18:36:01.098Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,-0.0005,NaN,NaN,NaN,NaN,NaN


In [7]:
markets_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Columns: 101 entries, id to groupItemRange
dtypes: bool(25), float64(27), int64(3), object(6), str(40)
memory usage: 2.3+ MB


In [6]:
markets_df.isnull().sum().sort_values(ascending=False).head(20)

eventStartTime         997
groupItemRange         993
secondsDelay           993
umaResolutionStatus    991
gameStartTime          969
positionIds            930
oneYearPriceChange     890
oneHourPriceChange     808
clobRewards            657
oneDayPriceChange      502
volume1moClob          424
volume1yrClob          424
volume1yr              424
volume1mo              424
volume1wk              424
volume24hr             424
volume24hrClob         424
volume1wkClob          424
negRiskMarketID        380
seriesColor            361
dtype: int64

In [8]:
market_columns = [
    "id",
    "question",
    "conditionId",
    "slug",
    "description",
    "startDate",
    "endDate",
    "active",
    "closed",
    "volume",
    "liquidity",
    "outcomes",
    "outcomePrices",
    "clobTokenIds",
]

In [9]:
[col for col in market_columns if col not in markets_df.columns]

[]

In [10]:
market_core_df = markets_df[market_columns].copy()

market_core_df.shape

(1000, 14)

In [11]:
print("Rows:", len(market_core_df))
print("Unique market IDs:", market_core_df["id"].nunique())
print("Unique condition IDs:", market_core_df["conditionId"].nunique())

Rows: 1000
Unique market IDs: 1000
Unique condition IDs: 1000


In [12]:
market_core_df[
    ["question", "outcomes", "outcomePrices", "clobTokenIds"]
].head(10)

,question,outcomes,outcomePrices,clobTokenIds
0,Xi Jinping out before 2027?,"[""Yes"", ""No""]","[""0.0375"", ""0.9625""]","[""32338220190071351435772801779725302244575775..."
1,Will Gavin Newsom win the 2028 Democratic pres...,"[""Yes"", ""No""]","[""0.1425"", ""0.8575""]","[""54533043819946592547517511176940999955633860..."
2,Will Alexandria Ocasio-Cortez win the 2028 Dem...,"[""Yes"", ""No""]","[""0.1955"", ""0.8045""]","[""10706498543549433311339103847040171911327280..."
3,Will Pete Buttigieg win the 2028 Democratic pr...,"[""Yes"", ""No""]","[""0.0565"", ""0.9435""]","[""91031279171981959197361710127213577102576826..."
4,Will Josh Shapiro win the 2028 Democratic pres...,"[""Yes"", ""No""]","[""0.0635"", ""0.9365""]","[""71169839497420596285841847514123589847893415..."
5,Will Wes Moore win the 2028 Democratic preside...,"[""Yes"", ""No""]","[""0.0155"", ""0.9845""]","[""90227223115596293448966158151606153337847900..."
6,Will Stephen A. Smith win the 2028 Democratic ...,"[""Yes"", ""No""]","[""0.0015"", ""0.9985""]","[""60590045489347122735554346200880179420435533..."
7,Will Kamala Harris win the 2028 Democratic pre...,"[""Yes"", ""No""]","[""0.0785"", ""0.9215""]","[""38171024903091354977195167437673496224420673..."
8,Will Gretchen Whitmer win the 2028 Democratic ...,"[""Yes"", ""No""]","[""0.0045"", ""0.9955""]","[""57761428076807364758801249497410455358987881..."
9,Will Andy Beshear win the 2028 Democratic pres...,"[""Yes"", ""No""]","[""0.0265"", ""0.9735""]","[""26468656392978559668331516709623917078428425..."


Trade Data Exploration

In [13]:
DATA_API = "https://data-api.polymarket.com"

In [14]:
all_trades = []

for offset in range(0, 1000, 100):
    response = requests.get(
        f"{DATA_API}/trades",
        params={
            "limit": 100,
            "offset": offset,
            "takerOnly": False
        },
        timeout=30
    )

    response.raise_for_status()

    batch = response.json()
    all_trades.extend(batch)

    print(f"Offset {offset}: {len(batch)} trades")

    if len(batch) < 100:
        break

    time.sleep(0.2)

print("Total trades collected:", len(all_trades))

Offset 0: 100 trades
Offset 100: 100 trades
Offset 200: 100 trades
Offset 300: 100 trades
Offset 400: 100 trades
Offset 500: 100 trades
Offset 600: 100 trades
Offset 700: 100 trades
Offset 800: 100 trades
Offset 900: 100 trades
Total trades collected: 1000


In [15]:
trades_df = pd.DataFrame(all_trades)

print("Shape:", trades_df.shape)

Shape: (1000, 19)


In [16]:
trades_df.columns.tolist()

['proxyWallet',
 'side',
 'asset',
 'conditionId',
 'size',
 'price',
 'timestamp',
 'title',
 'slug',
 'icon',
 'eventSlug',
 'outcome',
 'outcomeIndex',
 'name',
 'pseudonym',
 'bio',
 'profileImage',
 'profileImageOptimized',
 'transactionHash']

In [17]:
trades_df.head()

,proxyWallet,side,asset,conditionId,size,price,timestamp,title,slug,icon,eventSlug,outcome,outcomeIndex,name,pseudonym,bio,profileImage,profileImageOptimized,transactionHash
0,0xa3282d3e882501229c75d0caf134e62e3afb4977,BUY,3690252366244434109868507477168705473153569579...,0x5479759897d8e50a00e3cd4b292d6222089dccd224d0...,92.00,0.999,1789052551,Will Fernando Alonso win the 2026 F1 Spanish G...,f1-spanish-grand-prix-winner-alonso-2026-09-13,https://polymarket-upload.s3.us-east-2.amazona...,,No,999,korda77,Cheery-Leap-Pressurisation,,,,0x3af93574136341b8fa646906fc1c2713e9a3e8b09925...
1,0xf03d13c94373e052a177601419291ce2fafe21fb,BUY,1914168889765911990321430336258436455899955113...,0x5479759897d8e50a00e3cd4b292d6222089dccd224d0...,92.00,0.001,1789052551,Will Fernando Alonso win the 2026 F1 Spanish G...,f1-spanish-grand-prix-winner-alonso-2026-09-13,https://polymarket-upload.s3.us-east-2.amazona...,,Yes,999,0xF03D13c94373E052a177601419291Ce2FAfE21Fb-176...,Attached-Prairie,,,,0x3af93574136341b8fa646906fc1c2713e9a3e8b09925...
2,0xc69bd5567b40ef4d11922eaa57e1f9be1c642076,BUY,3333979837291603722078613613340647849111615062...,0xefa17dee3af09f69f9ddf245b969aa4efbe7c71cdf06...,5.00,0.002,1789052551,Will Shakhtar Donetsk win the 2026-27 UEFA Cha...,will-shakhtar-donetsk-win-the-2026-27-uefa-cha...,,,Yes,999,vtcchampion52,Ultimate-Underpass,grandmaster fiora,https://polymarket-upload.s3.us-east-2.amazona...,,0xddcbfde11f13f570416e72fd1ae8de32d0567b50a454...
3,0x3e9ff2dbf2a6356ee47049e9f3c43a70cb55d57f,SELL,3333979837291603722078613613340647849111615062...,0xefa17dee3af09f69f9ddf245b969aa4efbe7c71cdf06...,5.00,0.002,1789052551,Will Shakhtar Donetsk win the 2026-27 UEFA Cha...,will-shakhtar-donetsk-win-the-2026-27-uefa-cha...,,,Yes,999,vtcchampion53,Dismal-Gopher,Yes they will.,,,0xddcbfde11f13f570416e72fd1ae8de32d0567b50a454...
4,0x18eb2b1abb538ce9d76dfb88c52a16f196c71378,BUY,7247229568457909659163994513075518873557484069...,0x429a0bd966943512b23cb961db960e1de4c9bac2dbce...,25.05,0.998,1789052551,Will the highest temperature in Warsaw be 20°C...,highest-temperature-in-warsaw-on-september-10-...,https://polymarket-upload.s3.us-east-2.amazona...,,Yes,999,SjkyLaoGuo,Welcome-Powder-Lake,,,,0x42ddd4f6045422ae77f4ceb44b8ccbac00277c41ea8b...


In [18]:
print("Unique wallets:", trades_df["proxyWallet"].nunique())

Unique wallets: 487


In [19]:
print("Unique markets:", trades_df["conditionId"].nunique())

Unique markets: 107


In [20]:
print("Unique assets:", trades_df["asset"].nunique())

Unique assets: 200


In [21]:
trades_df["side"].value_counts()

side
BUY     829
SELL    171
Name: count, dtype: int64

In [22]:
trades_df["outcome"].value_counts()

outcome
Up                                 319
Down                               265
Yes                                135
No                                 119
GamerLegion                         27
Klim Sani4                          24
Over                                13
Under                               10
Teletubisie                          8
Revise                               8
                                     7
ShindeN                              6
Amsterdam Flames                     5
B8                                   5
Lourdes Ayala                        4
1WIN                                 4
Toby Martin                          4
Stefan Seifert                       4
BBL                                  3
Al Nassr Saudi Club                  3
Al Khaleej Saudi Club                3
3DMAX                                2
Belfast Wolves                       2
Enrico Dalla Valle                   2
Nemiga                               2
K27              

In [23]:
print("Earliest trade:", trades_df["timestamp"].min())
print("Latest trade:", trades_df["timestamp"].max())

Earliest trade: 1789052520
Latest trade: 1789052778


In [24]:
trades_df[["size", "price"]].describe()

,size,price
count,1000.000000,1000.000000
mean,46.445566,0.482569
std,216.512395,0.360990
min,0.020000,0.001000
25%,5.000000,0.070000
50%,8.013158,0.500000
75%,20.120000,0.800000
max,3127.910276,0.999000


In [25]:
trades_df["conditionId"].value_counts().head(10)

conditionId
0xce78c65bd527ca872a69bc712ee257cd3f72d7a68d191fd92fa57dbc2c89dad3    231
0xcfee480ec15754e25a44c826a626a5057d5d2eb864a9a5505348831e1fa0df63     76
0x5e65300019a0ce7c25f6f087101f8e73d1bbc18f9b364ec462530ce7d2735c92     56
0x9a4f6516e54f38d237526f1761cf75623bb5bfa6c515a7b6425615793ee49086     43
0x723f87ef04d9f13c2cabb70b4a1ab1c5e2230bdc3dc8a8fdcd0c09e2a2417c0f     40
0xa3c9e313edbe692554bf2a5277a8ad95dc0dc0adc0fcaa870d48d43715e92774     33
0xefa17dee3af09f69f9ddf245b969aa4efbe7c71cdf06ee49d694408bc33e2ed2     32
0x2daf6632f916badb6f81fe5dff6b7a95a5eefc8a602432b5510538ff3e703f17     21
0xcbbbeaf869f4d54c7249e635e53e9128584241871724e8bf2a174ea8ab68f14b     20
0xd2c635e1ae17b1eed6bbecfd58764daafe67e223df7725fd90ea314768253708     18
Name: count, dtype: int64

In [26]:
trades_df["proxyWallet"].value_counts().head(10)

proxyWallet
0x674887d1ac838099a48b629dff53f25b7b87ee08    30
0x322b2e7c44237c9fed01d534cd1c96687d52aa0c    19
0xc69bd5567b40ef4d11922eaa57e1f9be1c642076    16
0x9d57c42e847173d06841703825d3fe2299e456ea    16
0x21f4c1a1be34a39a7bc20985df4b7e5a39fb691d    16
0xa0e72d5575095199ae08ce0dce447f21b7249461    14
0x3e9ff2dbf2a6356ee47049e9f3c43a70cb55d57f    13
0xa0179c7be3fb25fbc6100fa4a5894710b74cac57    12
0xec10d2301a2333b75d1a809f7a164a2cf2566eb9    11
0x821dab0565ebf5b327f51db06223fdcfe01acf16    10
Name: count, dtype: int64

In [27]:
print(trades_df[[
    "timestamp",
    "proxyWallet",
    "conditionId",
    "side",
    "outcome",
    "size",
    "price"
]].head(10))

    timestamp                                 proxyWallet  \
0  1789052551  0xa3282d3e882501229c75d0caf134e62e3afb4977   
1  1789052551  0xf03d13c94373e052a177601419291ce2fafe21fb   
2  1789052551  0xc69bd5567b40ef4d11922eaa57e1f9be1c642076   
3  1789052551  0x3e9ff2dbf2a6356ee47049e9f3c43a70cb55d57f   
4  1789052551  0x18eb2b1abb538ce9d76dfb88c52a16f196c71378   
5  1789052551  0x32d1f0159dd59c734deebfcbf91db7c924fae5c8   
6  1789052551  0x1a0cec4e5ce0f4101eeaa8d37f3114871741e825   
7  1789052551  0x5900983386272bf5482f1de38cdb1ca9364b139c   
8  1789052551  0x61c647c2ca76b5bdf1d94192a6062b9d2348f212   
9  1789052551  0xa3282d3e882501229c75d0caf134e62e3afb4977   

                                         conditionId  side outcome    size  \
0  0x5479759897d8e50a00e3cd4b292d6222089dccd224d0...   BUY      No  92.000   
1  0x5479759897d8e50a00e3cd4b292d6222089dccd224d0...   BUY     Yes  92.000   
2  0xefa17dee3af09f69f9ddf245b969aa4efbe7c71cdf06...   BUY     Yes   5.000   
3  0xefa17dee3af

In [28]:
print(trades_df[[
    "timestamp",
    "proxyWallet",
    "conditionId",
    "side",
    "outcome",
    "size",
    "price"
]].tail(10))

      timestamp                                 proxyWallet  \
990  1789052520  0xb0f85baa97990910a3e8ac2b4a58a322f01ecef5   
991  1789052520  0x21f4c1a1be34a39a7bc20985df4b7e5a39fb691d   
992  1789052520  0xbf337426aa856996b8bb79b238345dd1a0276bf7   
993  1789052520  0xd4fa2e10c74ab94bc23414989a516b07386ddcc8   
994  1789052520  0x322b2e7c44237c9fed01d534cd1c96687d52aa0c   
995  1789052520  0xc387c2a40d389f17b723b6bba9b18b7dbd2de4f4   
996  1789052520  0x322b2e7c44237c9fed01d534cd1c96687d52aa0c   
997  1789052520  0x885278f0e304bc2d53f805af2ab779cb6011c569   
998  1789052520  0xb0f85baa97990910a3e8ac2b4a58a322f01ecef5   
999  1789052520  0xd4fa2e10c74ab94bc23414989a516b07386ddcc8   

                                           conditionId side outcome   size  \
990  0xce78c65bd527ca872a69bc712ee257cd3f72d7a68d19...  BUY    Down  20.00   
991  0xce78c65bd527ca872a69bc712ee257cd3f72d7a68d19...  BUY    Down  52.00   
992  0xce78c65bd527ca872a69bc712ee257cd3f72d7a68d19...  BUY    Down  15.